---

### Yang akan kita kerjakan

1. Membuat API key di NewsAPI.org
2. Mengambil data lewat API (dibungkus dalam class)
3. Mengubahnya jadi tabel
4. Membersihkan tabel itu sampai rapi
5. Menyimpan hasilnya

Tidak ada scraping di notebook ini, dan tidak ada penggabungan dua sumber. Cukup satu
sumber data yang benar-benar bersih di akhir.

Kalau lo memilih API lain untuk mini project lo sendiri, pola pengerjaannya tetap sama.
Yang berubah cuma alamat API dan nama kolom yang diambil.

---

# Bagian 1: Mendapatkan API Key

Sebelum bisa memanggil API-nya, kita perlu API key dulu. Ini langkah satu kali saja.

### Langkah membuat API key di NewsAPI.org

1. Buka [newsapi.org/register](https://newsapi.org/register)
2. Isi nama, email, dan password, lalu Submit
3. Setelah masuk, API key langsung tampil di halaman Account. Tidak perlu menunggu
   konfirmasi email dulu, key-nya langsung bisa dipakai.
4. Salin API key tersebut, panjangnya sekitar 32 karakter

Paket gratisnya (Developer) cukup untuk latihan kita: 100 kali panggilan per hari, dan
beritanya boleh yang agak lama (tidak harus breaking news hari ini).

### Menyimpan API key dengan aman

API key itu semacam password. Jangan ditulis langsung di dalam notebook, karena kalau
notebook ini nanti dikirim atau diunggah ke GitHub, orang lain bisa memakai key milik lo.

Caranya: simpan di file terpisah bernama `.env`, lalu panggil dari situ.

Buat file baru bernama `.env` di folder yang sama dengan notebook ini, isinya:

```
NEWS_API_KEY=isi_dengan_api_key_kamu
```

Simpan filenya, lalu lanjut ke cell berikutnya.

In [2]:
!pip install requests python-dotenv --quiet

In [4]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("NEWS_API_KEY")

if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


Kalau muncul tulisan "API key belum ketemu", periksa dua hal:

1. File `.env` ada di folder yang sama dengan notebook ini, bukan di folder lain
2. Tidak ada tanda kutip atau spasi di sekitar API key-nya di dalam file `.env`

Jangan lanjut ke cell berikutnya sebelum tulisan "berhasil dimuat" muncul.

---

# Bagian 2: Mencoba Memanggil API

### Mengenal alamat dan parameternya

NewsAPI punya beberapa alamat (disebut *endpoint*). Kita pakai yang bernama `everything`,
alamatnya:

```
https://newsapi.org/v2/everything
```

Sama seperti mengisi formulir permintaan di loket, kita perlu menyebutkan apa yang kita
cari lewat `params`:

- `q` : kata kunci beritanya
- `language` : bahasa berita
- `pageSize` : jumlah berita yang diminta sekaligus, maksimal 100
- `apiKey` : kartu identitas kita

### Kenapa harus dicoba dulu di satu panggilan kecil

Sebelum membuat class yang rapi, kita coba dulu satu panggilan sederhana. Tujuannya
melihat bentuk jawabannya dulu, sebelum menyusun kode yang lebih besar. Kalau langsung
loncat ke kode besar dan ternyata bentuk datanya beda dari yang dikira, akan lebih susah
mencari letak kesalahannya.

In [ ]:
alamat_api = "https://newsapi.org/v2/everything"

parameter = {
    "q"        : "teknologi",
    "language" : "id",
    "pageSize" : 5,
    "apiKey"   : API_KEY
}

response = requests.get(alamat_api, params=parameter)
print(f"Status Code: {response.status_code}")

hasil = response.json()
print(f"Jumlah berita ditemukan (totalResults): {hasil['totalResults']}")
print(f"Jumlah berita yang dikirim kali ini    : {len(hasil['articles'])}")

Status Code: 200
Jumlah berita ditemukan (totalResults): 40
Jumlah berita yang dikirim kali ini    : 5


In [6]:
# Lihat bentuk data satu berita
berita_pertama = hasil["articles"][0]
berita_pertama

{'source': {'id': None, 'name': 'Heathereatsalmondbutter.com'},
 'author': 'admin',
 'title': 'Pendekatan Berpikir Maju untuk casino premium',
 'description': 'Integrasi teknologi canggih ke dalam pengalaman digital sehari-hari merupakan frontier berikutnya dalam pengembangan platform dan keterlibatan pengguna. Teknologi progressive web application mengaburkan batas antara pengalaman native dan berbasis web, memberi…',
 'url': 'https://www.heathereatsalmondbutter.com/pendekatan-berpikir-maju-untuk-casino-premium/',
 'urlToImage': None,
 'publishedAt': '2026-09-09T03:00:00Z',
 'content': 'Integrasi teknologi canggih ke dalam pengalaman digital sehari-hari merupakan frontier berikutnya dalam pengembangan platform dan keterlibatan pengguna.\r\nTeknologi progressive web application mengabu… [+1898 chars]'}

Perhatikan bentuknya. Satu berita itu sebuah dictionary, isinya beberapa kunci seperti
`title`, `description`, `publishedAt`, `url`, dan `source`.

Yang agak berbeda adalah `source`. Isinya bukan tulisan biasa, tapi dictionary lagi di
dalam dictionary:

```python
"source": {"id": "some-id", "name": "Nama Media"}
```

Untuk mengambil nama medianya, caranya:

```python
berita_pertama["source"]["name"]
```

Kalau lo pakai API lain untuk mini project sendiri, kemungkinan besar akan ketemu bentuk
seperti ini juga. Cara mengambilnya sama: tinggal buka kurung siku sekali lagi.

In [7]:
print("Judul   :", berita_pertama["title"])
print("Sumber  :", berita_pertama["source"]["name"])
print("Tanggal :", berita_pertama["publishedAt"])

Judul   : Pendekatan Berpikir Maju untuk casino premium
Sumber  : Heathereatsalmondbutter.com
Tanggal : 2026-09-09T03:00:00Z


---

# Bagian 3: Membungkus Jadi Class

### Kenapa dibungkus jadi class

Ketentuan mini project meminta OOP diterapkan di bagian pengambilan data.

Bayangkan lo merekrut seorang petugas peliput berita. Petugas itu lo beri:

- **Kartu identitas yang dia pegang terus**, yaitu API key. Setiap kali dia bertugas,
  dia tidak perlu diberi tahu ulang identitasnya. Di dalam class, ini disimpan sebagai
  `self.api_key`.
- **Kemampuan yang bisa dia kerjakan**, yaitu mencari berita dengan kata kunci tertentu.
  Di dalam class, ini disebut **method**.

Isi method-nya nanti persis seperti kode yang barusan kita coba di atas. Tidak ada
konsep baru, cuma dipindah ke dalam kotak yang lebih rapi.

In [8]:
class KlienBerita:

    def __init__(self, api_key):
        # Kartu identitas disimpan di sini, supaya semua method di bawah bisa memakainya
        self.api_key = api_key
        self.alamat_api = "https://newsapi.org/v2/everything"

    def ambil_berita(self, kata_kunci, jumlah):
        parameter = {
            "q"        : kata_kunci,
            "language" : "id",
            "pageSize" : jumlah,
            "apiKey"   : self.api_key
        }

        try:
            # Rencana utama: menghubungi NewsAPI
            # timeout=20 artinya kita hanya sabar menunggu 20 detik
            response = requests.get(self.alamat_api, params=parameter, timeout=20)
        except Exception:
            # Rencana cadangan: kalau koneksi bermasalah, kita coba sekali lagi
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, timeout=20)

        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame()

        daftar_berita = response.json()["articles"]

        data = []
        for berita in daftar_berita:
            data.append({
                "Judul"        : berita["title"],
                "Deskripsi"    : berita["description"],
                "Sumber"       : berita["source"]["name"],
                "Tanggal"      : berita["publishedAt"],
                "URL"          : berita["url"]
            })

        return pd.DataFrame(data)


print("Class KlienBerita siap dipakai!")

Class KlienBerita siap dipakai!


Kenapa `try` dan `except` ditaruh di sini? Karena ini satu-satunya bagian yang menghubungi
internet. Bagian lain di dalam method ini, seperti mengambil `berita["title"]`, tidak
menghubungi internet, jadi tidak butuh pengaman itu.

Kalau koneksinya bermasalah, kita cukup coba sekali lagi. Kalau percobaan kedua juga
gagal, cell ini akan kasih tahu lewat tulisan "Gagal mengambil data", bukan berhenti
dengan tulisan error berwarna merah.

### Kenapa satu kata kunci saja tidak cukup untuk 100 baris

`pageSize` maksimal adalah 100, jadi sebenarnya satu panggilan saja sudah bisa mencapai
100 berita. Tapi kalau kata kuncinya terlalu sempit, jumlah berita yang benar-benar ada
mungkin kurang dari 100.

Supaya lebih aman, kita gabungkan beberapa kata kunci yang berhubungan, lalu satukan
hasilnya. Cara ini juga sekaligus jadi latihan menggabungkan beberapa tabel kecil jadi
satu tabel besar, dengan `pd.concat`.

In [9]:
klien = KlienBerita(API_KEY)

daftar_kata_kunci = ["teknologi", "ekonomi indonesia", "pendidikan"]

semua_tabel = []

for kata_kunci in daftar_kata_kunci:
    tabel = klien.ambil_berita(kata_kunci, jumlah=50)
    print(f"Kata kunci '{kata_kunci}': {len(tabel)} berita")
    semua_tabel.append(tabel)
    time.sleep(1)

df_berita = pd.concat(semua_tabel, ignore_index=True)

print()
print(f"Total berita terkumpul: {len(df_berita)}")
df_berita.head()

Kata kunci 'teknologi': 40 berita
Kata kunci 'ekonomi indonesia': 21 berita
Kata kunci 'pendidikan': 20 berita

Total berita terkumpul: 81


,Judul,Deskripsi,Sumber,Tanggal,URL
0,Pendekatan Berpikir Maju untuk casino premium,Integrasi teknologi canggih ke dalam pengalama...,Heathereatsalmondbutter.com,2026-09-09T03:00:00Z,https://www.heathereatsalmondbutter.com/pendek...
1,Perbedaan Slot Online dan Mesin Slot Konvensional,Slot online dan mesin slot konvensional memili...,Heathereatsalmondbutter.com,2026-09-13T06:41:51Z,https://www.heathereatsalmondbutter.com/perbed...
2,Bagaimana Umpan Balik Meningkatkan cashback ca...,Penekanan pada keamanan dan kepercayaan telah ...,Heathereatsalmondbutter.com,2026-09-03T18:07:49Z,https://www.heathereatsalmondbutter.com/bagaim...
3,Analisis Mendalam Tentang Tren togel Sydney Te...,Perkembangan teknologi telah membawa transform...,Heathereatsalmondbutter.com,2026-08-27T03:00:00Z,https://www.heathereatsalmondbutter.com/analis...
4,Strategi Analisis bandar togel Berdasarkan Dat...,Analisis berbasis data telah mengubah cara pem...,Heathereatsalmondbutter.com,2026-09-06T03:00:00Z,https://www.heathereatsalmondbutter.com/strate...


Ketentuan mini project meminta minimal 100 baris. Pastikan angka di atas sudah tercapai.

Kalau kurang, tambahkan kata kunci lain ke `daftar_kata_kunci`, lalu jalankan ulang.

---

# Bagian 4: Membersihkan Data

Ini bagian dengan bobot nilai paling besar. Tiga hal yang wajib ditangani:

1. Nilai yang kosong
2. Baris yang kembar
3. Tipe data yang tidak sesuai

Aturan mainnya sama seperti biasa: **lihat dulu kondisinya, baru putuskan tindakannya.**

In [10]:
print("1. Jumlah sel kosong per kolom:")
print(df_berita.isnull().sum())
print()

print("2. Jumlah baris yang kembar (berdasarkan URL):")
print(df_berita.duplicated(subset="URL").sum())
print()

print("3. Tipe data setiap kolom:")
print(df_berita.dtypes)

1. Jumlah sel kosong per kolom:
Judul        0
Deskripsi    0
Sumber       0
Tanggal      0
URL          0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan URL):
10

3. Tipe data setiap kolom:
Judul        str
Deskripsi    str
Sumber       str
Tanggal      str
URL          str
dtype: object


### Membaca hasil pemeriksaan

Kemungkinan besar ada beberapa berita yang `Deskripsi`-nya kosong. Ini wajar, tidak semua
media menulis deskripsi untuk setiap beritanya.

Baris kembar bisa muncul karena kata kunci yang berbeda kadang menemukan berita yang sama.
Kita cek berdasarkan `URL`, bukan `Judul`, karena URL pasti unik untuk setiap berita,
sedangkan judul yang mirip belum tentu berita yang sama persis.

Kolom `Tanggal` kemungkinan masih bertipe `object` (tulisan), padahal isinya tanggal.
Ini perlu diubah supaya nanti bisa diurutkan dengan benar berdasarkan waktu.

### Menangani nilai kosong

| Kolom | Tindakan | Alasan |
|---|---|---|
| Deskripsi | Isi dengan "Tidak ada deskripsi" | Bukan angka yang dihitung, jadi aman diberi penanda |
| Judul | Buang barisnya kalau kosong | Berita tanpa judul tidak berguna untuk dianalisis |

Function di bawah memakai `pd.isna(teks)`, bukan `teks is None` atau `teks == None`.
Alasannya, sel kosong di dalam kolom pandas kadang muncul dalam bentuk `None`, kadang
dalam bentuk `NaN`, tergantung tipe kolomnya. `pd.isna()` mengenali kedua bentuk itu
sekaligus, jadi lebih aman dipakai daripada menebak salah satu bentuknya saja.

In [11]:
def bersihkan_deskripsi(teks):
    if pd.isna(teks):
        return "Tidak ada deskripsi"
    return teks


df_berita["Deskripsi"] = df_berita["Deskripsi"].apply(bersihkan_deskripsi)

# Baris tanpa judul kita buang, karena berita seperti itu tidak berguna
jumlah_sebelum = len(df_berita)
df_berita = df_berita.dropna(subset=["Judul"])
print(f"Baris tanpa judul yang dibuang: {jumlah_sebelum - len(df_berita)}")

print()
print("Sel kosong setelah ditangani:")
print(df_berita.isnull().sum())

Baris tanpa judul yang dibuang: 0

Sel kosong setelah ditangani:
Judul        0
Deskripsi    0
Sumber       0
Tanggal      0
URL          0
dtype: int64


### Menangani baris kembar

In [12]:
jumlah_sebelum = len(df_berita)

df_bersih = df_berita.drop_duplicates(subset="URL")

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")

Jumlah baris sebelum : 81
Jumlah baris sesudah : 71
Baris kembar dibuang : 10


### Menangani tipe data

Kolom `Tanggal` datang dalam bentuk tulisan seperti `2026-09-15T10:30:00Z`. Supaya bisa
diurutkan dan dihitung selisih harinya, kita ubah jadi tipe tanggal sungguhan memakai
`pd.to_datetime`.

In [13]:
def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks)


print("Tipe data sebelum:", df_bersih["Tanggal"].dtype)

df_bersih["Tanggal"] = df_bersih["Tanggal"].apply(ubah_ke_tanggal)

print("Tipe data sesudah:", df_bersih["Tanggal"].dtype)
df_bersih.head()

Tipe data sebelum: str
Tipe data sesudah: datetime64[us, UTC]


,Judul,Deskripsi,Sumber,Tanggal,URL
0,Pendekatan Berpikir Maju untuk casino premium,Integrasi teknologi canggih ke dalam pengalama...,Heathereatsalmondbutter.com,2026-09-09 03:00:00+00:00,https://www.heathereatsalmondbutter.com/pendek...
1,Perbedaan Slot Online dan Mesin Slot Konvensional,Slot online dan mesin slot konvensional memili...,Heathereatsalmondbutter.com,2026-09-13 06:41:51+00:00,https://www.heathereatsalmondbutter.com/perbed...
2,Bagaimana Umpan Balik Meningkatkan cashback ca...,Penekanan pada keamanan dan kepercayaan telah ...,Heathereatsalmondbutter.com,2026-09-03 18:07:49+00:00,https://www.heathereatsalmondbutter.com/bagaim...
3,Analisis Mendalam Tentang Tren togel Sydney Te...,Perkembangan teknologi telah membawa transform...,Heathereatsalmondbutter.com,2026-08-27 03:00:00+00:00,https://www.heathereatsalmondbutter.com/analis...
4,Strategi Analisis bandar togel Berdasarkan Dat...,Analisis berbasis data telah mengubah cara pem...,Heathereatsalmondbutter.com,2026-09-06 03:00:00+00:00,https://www.heathereatsalmondbutter.com/strate...


In [14]:
# Pemeriksaan terakhir sebelum dianggap selesai

print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Judul masih ada kosong : {df_bersih['Judul'].isnull().sum()}")
print(f"URL masih kembar       : {df_bersih['URL'].duplicated().sum()}")
print(f"Tipe kolom Tanggal     : {df_bersih['Tanggal'].dtype}")

Jumlah baris           : 71
Sudah lebih dari 100?  : False
Judul masih ada kosong : 0
URL masih kembar       : 0
Tipe kolom Tanggal     : datetime64[us, UTC]


---

# Bagian 5: Menyimpan Hasil

File CSV inilah yang dikumpulkan bersama notebook.

In [15]:
df_bersih.to_csv("dataset_berita.csv", index=False)
print("Data berhasil disimpan ke file: dataset_berita.csv")

df_cek = pd.read_csv("dataset_berita.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_berita.csv
File terbaca kembali: 71 baris, 5 kolom


,Judul,Deskripsi,Sumber,Tanggal,URL
0,Pendekatan Berpikir Maju untuk casino premium,Integrasi teknologi canggih ke dalam pengalama...,Heathereatsalmondbutter.com,2026-09-09 03:00:00+00:00,https://www.heathereatsalmondbutter.com/pendek...
1,Perbedaan Slot Online dan Mesin Slot Konvensional,Slot online dan mesin slot konvensional memili...,Heathereatsalmondbutter.com,2026-09-13 06:41:51+00:00,https://www.heathereatsalmondbutter.com/perbed...
2,Bagaimana Umpan Balik Meningkatkan cashback ca...,Penekanan pada keamanan dan kepercayaan telah ...,Heathereatsalmondbutter.com,2026-09-03 18:07:49+00:00,https://www.heathereatsalmondbutter.com/bagaim...
3,Analisis Mendalam Tentang Tren togel Sydney Te...,Perkembangan teknologi telah membawa transform...,Heathereatsalmondbutter.com,2026-08-27 03:00:00+00:00,https://www.heathereatsalmondbutter.com/analis...
4,Strategi Analisis bandar togel Berdasarkan Dat...,Analisis berbasis data telah mengubah cara pem...,Heathereatsalmondbutter.com,2026-09-06 03:00:00+00:00,https://www.heathereatsalmondbutter.com/strate...


---

# Bagian 6: Bahan untuk Slide

Slide presentasi tetap wajib dikumpulkan. Cell di bawah mencetak angka yang bisa
langsung disalin ke slide.

In [16]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)
print(f"Sumber data      : NewsAPI.org")
print(f"Kata kunci dipakai: {', '.join(daftar_kata_kunci)}")
print()
print(f"Baris sebelum dibersihkan : {len(df_berita) + (jumlah_sebelum - len(df_bersih))}")
print(f"Baris dataset akhir       : {len(df_bersih)}")
print()
print("Class yang dibuat:")
print("  1. KlienBerita - mengambil data berita dari NewsAPI")
print()
print("Function yang dibuat:")
print("  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda")
print("  2. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal")
print()
print("Temuan dari pembersihan data:")
print(f"  Baris kembar dibuang     : {jumlah_sebelum - len(df_bersih)}")
print(f"  Deskripsi kosong diberi penanda: ada")
print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data      : NewsAPI.org
Kata kunci dipakai: teknologi, ekonomi indonesia, pendidikan

Baris sebelum dibersihkan : 91
Baris dataset akhir       : 71

Class yang dibuat:
  1. KlienBerita - mengambil data berita dari NewsAPI

Function yang dibuat:
  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda
  2. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal

Temuan dari pembersihan data:
  Baris kembar dibuang     : 10
  Deskripsi kosong diberi penanda: ada


---

# Ringkasan

| Aspek | Ada di bagian |
|---|---|
| Pengambilan data lewat API | Bagian 2 dan 3 |
| Struktur OOP | Bagian 3, class `KlienBerita` |
| Function dan modularitas | Bagian 4, `bersihkan_deskripsi` dan `ubah_ke_tanggal` |
| Data cleaning | Bagian 4 |
| Minimal 100 baris | Dicek di akhir Bagian 3 dan Bagian 4 |

### Kalau memilih API lain untuk mini project sendiri

Yang berubah cuma dua tempat:

1. **Alamat API dan parameter di dalam `ambil_berita`.** Ganti dengan alamat API pilihan
   lo, dan sesuaikan nama parameternya (biasanya ada di dokumentasi API tersebut).

2. **Nama kolom yang diambil dari jawaban API.** Sesuaikan dengan bentuk data yang
   dikembalikan API pilihan lo.

Sisanya, mulai dari cara membuat class, cara membersihkan data, sampai cara menyimpan
file, polanya sama untuk API apa pun.